# Chatbots e APIs no Google Colab

Rode as células **de cima para baixo** (Shift + Enter).

| Parte | O que é |
|---|---|
| 1 | Preparar o ambiente |
| 2 | **Chatbot do clima**: um chatbot que *consome* uma API real (Open-Meteo, sem chave) |
| 3 | **Chatbot como API** (Flask): um chatbot que *é* uma API |
| 4 | Dicas para rodar os exercícios do professor |

## 1. Preparar o ambiente
O Colab já tem quase tudo instalado; esta célula só garante.

In [ ]:
!pip install -q flask nltk requests

## 2. Chatbot do clima (consome uma API)

A célula abaixo **salva** o código no arquivo `chatbot_clima.py` (o comando `%%writefile` faz isso).
Os comentários dentro do código explicam cada parte.

In [ ]:
%%writefile chatbot_clima.py
"""
=====================================================================
 CHATBOT DO CLIMA: exemplo de chatbot que USA UMA API
=====================================================================
É a mesma ideia dos exercícios do professor, só que menor:

  Exercício 1 (turístico)  ->  cidade -> coordenadas -> dados da API
  Exercício 2 (pedidos)    ->  intenção, entidade e sentimento por regras

API usada: Open-Meteo (https://open-meteo.com). É gratuita e NÃO precisa
de chave, então dá para rodar direto.

Camadas do livro (Aula 1), e onde cada uma está no código:
  1. INTERFACE ........... conversar()           (input / print)
  2. NLP (regras) ........ identificar_intencao(), extrair_cidade(),
                           analisar_sentimento()
  3. LÓGICA DE NEGÓCIOS .. gerar_resposta() + buscar_coordenadas()
                           + buscar_clima()     (é aqui que a API é chamada)
  4. RESPOSTA ............ o texto que gerar_resposta() devolve

Rodar:
    pip install requests
    python chatbot_clima.py

Exemplos de pergunta:
    Oi
    Como está o tempo em São Paulo?
    Vai chover no Rio de Janeiro?
    Qual a temperatura em Xyzabc?        (cidade que não existe)
    Esse bot é péssimo                   (sentimento negativo -> humano)
"""
import re

import requests

# Endereços (URLs) das duas rotas da API que vamos usar
URL_GEOCODING = "https://geocoding-api.open-meteo.com/v1/search"  # nome -> coordenadas
URL_CLIMA = "https://api.open-meteo.com/v1/forecast"              # coordenadas -> clima


# ---------------------------------------------------------------------
# CAMADA 2: NLP BASEADO EM REGRAS
# ---------------------------------------------------------------------

def tokenizar(texto):
    """Quebra o texto em palavras, já SEM pontuação.

    No exercício 2, o split() deixava 'oi,' e 'bem?' com pontuação.
    Aqui o re.findall(r"\\w+") pega só letras e números:
        "Oi, tudo bem?"  ->  ['oi', 'tudo', 'bem']
    """
    return re.findall(r"\w+", texto.lower())


def identificar_intencao(texto):
    """INTENÇÃO = o que o usuário quer fazer.

    Comparamos com os TOKENS (palavras inteiras), não com o texto todo.
    Assim 'dois' e 'noite' NÃO viram saudação (pegadinha do exercício 2).
    A ordem dos if importa: o primeiro que combinar vence.
    """
    palavras = tokenizar(texto)

    if any(p in palavras for p in ["atendente", "humano"]):
        return "falar_com_humano"
    if any(p in palavras for p in ["tempo", "clima", "temperatura", "chover", "chuva"]):
        return "consultar_clima"
    if any(p in palavras for p in ["oi", "olá", "ola"]):
        return "saudacao"
    return "desconhecida"


def extrair_cidade(texto):
    """ENTIDADE = a informação específica que completa a intenção (a cidade).

    1ª tentativa: palavras com letra MAIÚSCULA depois de em/no/na.
        "tempo em São Paulo amanhã?"        -> "São Paulo"
        "Vai chover no Rio de Janeiro?"     -> "Rio de Janeiro"
        "tempo em Paris no inverno"         -> "Paris"
        (no exercício 1 dava "Paris no inverno")
    2ª tentativa: se escreveu tudo minúsculo, pega só a palavra seguinte.
        "tempo em curitiba"  -> "curitiba"
    """
    maiusculas = re.search(
        r"\b(?:em|no|na)\s+([A-ZÀ-Ú]\w*(?:\s+(?:d[aeo]s?\s+)?[A-ZÀ-Ú]\w*)*)",
        texto,
    )
    if maiusculas:
        return maiusculas.group(1)

    qualquer = re.search(r"\b(?:em|no|na)\s+(\w+)", texto, re.IGNORECASE)
    if qualquer:
        return qualquer.group(1)

    return None


def analisar_sentimento(texto):
    """Sentimento por lista de palavras (regras, como no exercício 2)."""
    palavras = tokenizar(texto)
    negativas = ["ruim", "péssimo", "pessimo", "horrível", "frustrado", "irritado", "inútil"]
    positivas = ["ótimo", "otimo", "obrigado", "obrigada", "excelente", "valeu"]

    if any(p in palavras for p in negativas):
        return "negativo"
    if any(p in palavras for p in positivas):
        return "positivo"
    return "neutro"


# ---------------------------------------------------------------------
# CAMADA 3: LÓGICA DE NEGÓCIOS (integração com a API)
# ---------------------------------------------------------------------

def buscar_coordenadas(cidade):
    """1ª CHAMADA À API: GET /v1/search  (nome da cidade -> latitude/longitude)

    É o mesmo papel do /geoname do exercício 1.
    """
    resposta = requests.get(
        URL_GEOCODING,
        params={"name": cidade, "count": 1, "language": "pt"},  # vira ?name=...&count=1&language=pt
        timeout=10,                 # se a API não responder em 10 s, desiste (não trava)
    )
    resposta.raise_for_status()     # erro HTTP 4xx/5xx -> lança exceção
    dados = resposta.json()         # texto JSON -> dicionário Python

    # Quando a cidade não existe, a API NÃO manda a chave "results".
    # Por isso usamos .get(): devolve None em vez de dar KeyError.
    resultados = dados.get("results")
    if not resultados:
        return None

    return resultados[0]  # ex.: {"name": "São Paulo", "latitude": -23.5, "longitude": -46.6, "country": "Brasil", ...}


def buscar_clima(latitude, longitude):
    """2ª CHAMADA À API: GET /v1/forecast  (coordenadas -> clima atual)

    É o mesmo papel do /radius do exercício 1: só funciona com lat/lon,
    por isso precisa da 1ª chamada antes (encadeamento de APIs).
    """
    resposta = requests.get(
        URL_CLIMA,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,precipitation,wind_speed_10m",  # o que queremos
            "timezone": "auto",
        },
        timeout=10,
    )
    resposta.raise_for_status()
    return resposta.json()["current"]  # ex.: {"temperature_2m": 25.7, "precipitation": 0.0, ...}


def dica(temperatura, chuva):
    """Faixas de valores, como as faixas de pedido do exercício 2."""
    if chuva > 0:
        return "Leve guarda-chuva!"
    if temperatura >= 30:
        return "Está bem quente, beba água."
    if temperatura <= 15:
        return "Está frio, leve um casaco."
    return "Clima agradável."


def gerar_resposta(texto):
    intencao = identificar_intencao(texto)
    sentimento = analisar_sentimento(texto)

    # ESCALONAMENTO PARA HUMANO (livro, Aula 2). Gatilhos:
    #   - palavra-chave (pediu atendente)
    #   - feedback negativo do usuário
    #   - o bot não tem uma resposta clara (intenção desconhecida)
    # Aqui o sentimento É USADO (no exercício 2 ele era calculado e ignorado).
    if intencao == "falar_com_humano" or sentimento == "negativo":
        return "Sinto muito! Vou te transferir para um atendente humano."
    if intencao == "desconhecida":
        return ("Não entendi. Pergunte, por exemplo: 'Como está o tempo em Recife?'. "
                "Se preferir, digite 'atendente'.")

    if intencao == "saudacao":
        return "Olá! Pergunte sobre o clima de alguma cidade."

    # intenção == "consultar_clima"
    cidade = extrair_cidade(texto)
    if not cidade:
        return "De qual cidade você quer saber o clima? Ex.: 'tempo em Salvador'."

    local = buscar_coordenadas(cidade)
    if local is None:
        return f"Não encontrei a cidade '{cidade}'."

    clima = buscar_clima(local["latitude"], local["longitude"])
    temperatura = clima["temperature_2m"]
    chuva = clima["precipitation"]

    # CAMADA 4: RESPOSTA
    return (f"{local['name']} ({local.get('country', '?')}): {temperatura} °C, "
            f"chuva {chuva} mm, vento {clima['wind_speed_10m']} km/h. "
            f"{dica(temperatura, chuva)}")


# ---------------------------------------------------------------------
# CAMADA 1: INTERFACE
# ---------------------------------------------------------------------

def conversar():
    print("=== CHATBOT DO CLIMA ===  (digite 'sair' para encerrar)")

    while True:
        mensagem = input("\nVocê: ").strip()   # strip(): "sair " também encerra

        if mensagem.lower() == "sair":
            print("Bot: Até mais!")
            break

        # Os except vão do MAIS ESPECÍFICO para o MAIS GENÉRICO.
        # Timeout é um tipo de RequestException, então precisa vir antes.
        try:
            print("Bot:", gerar_resposta(mensagem))
        except requests.exceptions.Timeout:
            print("Bot: A API do clima demorou demais. Tente de novo.")
        except requests.exceptions.RequestException as erro:
            print("Bot: Erro ao consultar a API:", erro)
        except Exception as erro:
            print("Bot: Ocorreu um erro inesperado:", erro)


# Só roda o chat quando o arquivo é executado direto.
# Se outro arquivo fizer "import chatbot_clima" (ex.: os testes), não abre o input().
if __name__ == "__main__":
    conversar()

### 2.1 Testes (rodam sem internet, porque a API é simulada com `unittest.mock`)

In [ ]:
%%writefile test_clima.py
"""Testes do chatbot do clima.  Rodar: python -m unittest test_clima -v

Usamos unittest.mock para FINGIR a API. Assim o teste:
  - não precisa de internet
  - sempre recebe a mesma resposta (resultado previsível)
"""
import unittest
from unittest.mock import MagicMock, patch

import chatbot_clima as bot


def resposta_falsa(json_dict):
    """Cria um objeto que se comporta como a resposta do requests."""
    falsa = MagicMock()
    falsa.json.return_value = json_dict
    falsa.raise_for_status.return_value = None
    return falsa


class TestRegras(unittest.TestCase):
    def test_tokenizar_tira_pontuacao(self):
        self.assertEqual(bot.tokenizar("Oi, tudo bem?"), ["oi", "tudo", "bem"])

    def test_dois_nao_e_saudacao(self):
        self.assertEqual(bot.identificar_intencao("Quero dois guarda-chuvas"), "desconhecida")

    def test_intencao_clima(self):
        self.assertEqual(bot.identificar_intencao("Vai chover amanhã?"), "consultar_clima")

    def test_extrair_cidade_composta(self):
        self.assertEqual(bot.extrair_cidade("Vai chover no Rio de Janeiro?"), "Rio de Janeiro")

    def test_extrair_cidade_sem_pegar_o_resto(self):
        self.assertEqual(bot.extrair_cidade("tempo em Paris no inverno"), "Paris")

    def test_extrair_cidade_minuscula(self):
        self.assertEqual(bot.extrair_cidade("tempo em curitiba"), "curitiba")

    def test_sentimento_negativo_escala(self):
        self.assertIn("atendente humano", bot.gerar_resposta("esse bot é péssimo"))


class TestAPI(unittest.TestCase):
    @patch("chatbot_clima.requests.get")
    def test_consulta_completa(self, get_falso):
        # 1ª chamada (geocoding) e 2ª chamada (clima), nessa ordem
        get_falso.side_effect = [
            resposta_falsa({"results": [{"name": "Recife", "country": "Brasil",
                                         "latitude": -8.05, "longitude": -34.9}]}),
            resposta_falsa({"current": {"temperature_2m": 31.0, "precipitation": 0.0,
                                        "wind_speed_10m": 12.0}}),
        ]
        texto = bot.gerar_resposta("Como está o tempo em Recife?")
        self.assertIn("Recife (Brasil): 31.0 °C", texto)
        self.assertIn("bem quente", texto)
        self.assertEqual(get_falso.call_count, 2)  # chamou a API duas vezes

    @patch("chatbot_clima.requests.get")
    def test_cidade_inexistente(self, get_falso):
        get_falso.return_value = resposta_falsa({"generationtime_ms": 0.1})  # sem "results"
        self.assertEqual(bot.gerar_resposta("tempo em Xyzabc"), "Não encontrei a cidade 'Xyzabc'.")


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest test_clima -v

### 2.2 Uma pergunta de cada vez (usa a API de verdade)

In [ ]:
from chatbot_clima import gerar_resposta

for pergunta in ["Oi", "Como está o tempo em São Paulo?", "Vai chover no Rio de Janeiro?",
                 "Qual a temperatura em Xyzabc?", "Esse bot é péssimo"]:
    print("Você:", pergunta)
    print("Bot: ", gerar_resposta(pergunta), "\n")

### 2.3 Conversar (digite na caixa que aparece; `sair` encerra)

In [ ]:
from chatbot_clima import conversar
conversar()

## 3. Chatbot como API (Flask)

**Atenção:** no Colab, `app.run()` deixa a célula rodando para sempre e não gera um link que dê para abrir.
Por isso usamos o `app.test_client()`: ele simula as requisições HTTP (GET, POST, DELETE) sem subir o servidor.

In [ ]:
%%writefile chatbot_api.py
"""
Mini projeto: API de um chatbot BASEADO EM REGRAS (livro U4 + slides).

Segue a arquitetura em camadas do livro (Aula 1):
  1. Interface ............ as rotas HTTP (o site/app/WhatsApp chamam a API)
  2. Processamento (NLP) .. os padrões de regras do NLTK (Chat)
  3. Lógica de negócios ... consultar o "CRM" de pedidos, decidir escalonar
  4. Resposta ............. o JSON devolvido ao usuário

Também tem:
  - Escalonamento para humano quando o bot não entende (Aula 2)
  - Testes com unittest (Aula 3)
  - Token de acesso, anonimização, auditoria e exclusão de dados (Aula 4)
  - KPIs: Engajamento, Conclusão, CSAT e Transbordo (slide "Medindo o Sucesso")

Rodar:  python chatbot_api.py   (abre em http://127.0.0.1:5000)
"""
import hashlib
import re

from flask import Flask, jsonify, request
from nltk.chat.util import Chat

# ---------------------------------------------------------------
# CAMADA 2: PROCESSAMENTO. Regras (o mesmo exemplo do slide, com mais pares)
# ---------------------------------------------------------------
pares = [
    [r"oi|olá", ["Olá, como posso ajudar?"]],
    [r"tchau", ["Até mais!"]],
    [r"(.*)horário(.*)", ["Atendemos de segunda a sexta, das 8h às 18h."]],
    [r"(.*)devol(.*)", ["Você pode devolver o produto em até 7 dias."]],
]
chatbot = Chat(pares)

# Palavras que pedem atendente na hora (gatilho de escalonamento, Aula 2)
PALAVRAS_ESCALONAMENTO = ["atendente", "humano", "reclamação"]

# ---------------------------------------------------------------
# CAMADA 3: LÓGICA DE NEGÓCIOS. "CRM" simulado (na vida real, outra API)
# ---------------------------------------------------------------
CRM = {
    "1001": {"cliente": "Ana", "status": "Enviado"},
    "1002": {"cliente": "Bruno", "status": "Em separação"},
}

# Controle de acesso: só a gestão pode ver as métricas (Aula 4)
TOKEN_GESTAO = "segredo123"

dados = {}


def resetar():
    """Zera os contadores (usado nos testes)."""
    dados.update(
        visitantes=0,          # quantas pessoas entraram no site
        usuarios_chat=set(),   # quem usou o chat (anonimizado, sem repetir)
        interacoes=0,          # mensagens recebidas = tarefas iniciadas
        resolvidas=0,          # o bot respondeu sozinho = tarefas OK
        transferidas=0,        # foi para um humano (transbordo/escalonamento)
        notas=[],              # notas de 1 a 5 (CSAT)
        auditoria=[],          # registro de cada interação (Aula 4)
    )


def anonimizar(usuario):
    """Guarda um código no lugar do nome (anonimização, Aula 4)."""
    return hashlib.sha256(usuario.encode()).hexdigest()[:10]


def registrar(usuario_anon, texto, transferido):
    dados["auditoria"].append(
        {"usuario": usuario_anon, "texto": texto, "transferido": transferido}
    )


def escalonar(usuario_anon, texto):
    dados["transferidas"] += 1
    registrar(usuario_anon, texto, True)
    return jsonify({"resposta": "Vou te transferir para um atendente humano.",
                    "transferido": True}), 200


def responder(usuario_anon, texto, resposta):
    dados["resolvidas"] += 1
    registrar(usuario_anon, texto, False)
    return jsonify({"resposta": resposta, "transferido": False}), 200


resetar()
app = Flask(__name__)
app.json.ensure_ascii = False  # mostra "Olá" em vez de "Olá"


# ---------------------------------------------------------------
# CAMADA 1: INTERFACE. As rotas (endpoints) da API
# ---------------------------------------------------------------
@app.get("/status")
def status():
    return jsonify({"status": "online"}), 200


@app.post("/visita")
def registrar_visita():
    dados["visitantes"] += 1
    return jsonify({"visitantes": dados["visitantes"]}), 201


@app.post("/mensagem")
def mensagem():
    corpo = request.get_json(silent=True) or {}
    texto = corpo.get("texto", "").strip()
    usuario = anonimizar(corpo.get("usuario", "anonimo"))

    if not texto:
        return jsonify({"erro": "Campo 'texto' é obrigatório"}), 400

    dados["usuarios_chat"].add(usuario)
    dados["interacoes"] += 1
    minusculo = texto.lower()

    # 1) Pediu humano explicitamente? Escala na hora.
    if any(p in minusculo for p in PALAVRAS_ESCALONAMENTO):
        return escalonar(usuario, texto)

    # 2) Falou de pedido? Lógica de negócios: consulta o CRM (integração via API)
    achou = re.search(r"pedido (\d+)", minusculo)
    if achou:
        pedido = CRM.get(achou.group(1))
        if pedido:
            return responder(usuario, texto, f"Seu pedido está: {pedido['status']}")

    # 3) Regras do NLTK
    resposta = chatbot.respond(texto)
    if resposta is None:  # nenhuma regra casou: limitação do bot de regras
        return escalonar(usuario, texto)

    # CAMADA 4: RESPOSTA
    return responder(usuario, texto, resposta)


@app.get("/pedidos/<numero>")
def consultar_pedido(numero):
    pedido = CRM.get(numero)
    if pedido is None:
        return jsonify({"erro": "Pedido não encontrado"}), 404
    return jsonify(pedido), 200


@app.post("/avaliacao")
def avaliacao():
    corpo = request.get_json(silent=True) or {}
    nota = corpo.get("nota")
    if not isinstance(nota, int) or not 1 <= nota <= 5:
        return jsonify({"erro": "A nota deve ser um número de 1 a 5"}), 400
    dados["notas"].append(nota)
    return jsonify({"mensagem": "Obrigado pela avaliação!"}), 201


@app.delete("/usuarios/<usuario>")
def excluir_dados(usuario):
    """Direito do usuário de excluir seus dados (privacidade/LGPD, Aula 4)."""
    anon = anonimizar(usuario)
    dados["auditoria"] = [r for r in dados["auditoria"] if r["usuario"] != anon]
    return "", 204


def porcentagem(parte, total):
    return round(parte / total * 100, 1) if total else 0.0


@app.get("/kpis")
def kpis():
    if request.headers.get("Authorization") != f"Bearer {TOKEN_GESTAO}":
        return jsonify({"erro": "Não autorizado"}), 401

    notas = dados["notas"]
    return jsonify({
        # (Usuários Chat / Total Visitantes) * 100
        "engajamento": porcentagem(len(dados["usuarios_chat"]), dados["visitantes"]),
        # (Tarefas OK / Tarefas Iniciadas) * 100
        "conclusao": porcentagem(dados["resolvidas"], dados["interacoes"]),
        # Média das notas pós-atendimento
        "csat": round(sum(notas) / len(notas), 1) if notas else 0.0,
        # (Transf. Humano / Total Interações) * 100
        "transbordo": porcentagem(dados["transferidas"], dados["interacoes"]),
    }), 200


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
%%writefile test_chatbot.py
"""Testes da Aula 3 (etapa 3: Teste — Unittest).  Rodar: python -m unittest -v"""
import unittest

from chatbot_api import app, dados, resetar

TOKEN = {"Authorization": "Bearer segredo123"}


class TestChatbotAPI(unittest.TestCase):
    def setUp(self):
        resetar()                       # cada teste começa do zero
        self.cliente = app.test_client()  # "navegador" de mentira, sem subir servidor

    def test_status_online(self):
        r = self.cliente.get("/status")
        self.assertEqual(r.status_code, 200)
        self.assertEqual(r.get_json(), {"status": "online"})

    def test_saudacao(self):
        r = self.cliente.post("/mensagem", json={"texto": "oi"})
        self.assertEqual(r.get_json()["resposta"], "Olá, como posso ajudar?")

    def test_despedida(self):
        r = self.cliente.post("/mensagem", json={"texto": "tchau"})
        self.assertEqual(r.get_json()["resposta"], "Até mais!")

    def test_mensagem_sem_texto_da_400(self):
        r = self.cliente.post("/mensagem", json={})
        self.assertEqual(r.status_code, 400)

    def test_nao_entendeu_transfere_para_humano(self):
        r = self.cliente.post("/mensagem", json={"texto": "quero cancelar meu cartão"})
        self.assertTrue(r.get_json()["transferido"])

    def test_consulta_pedido_no_crm(self):
        r = self.cliente.post("/mensagem", json={"texto": "Cadê meu pedido 1001?"})
        self.assertEqual(r.get_json()["resposta"], "Seu pedido está: Enviado")

    def test_pedido_inexistente_da_404(self):
        self.assertEqual(self.cliente.get("/pedidos/9999").status_code, 404)

    def test_avaliacao_invalida_da_400(self):
        r = self.cliente.post("/avaliacao", json={"nota": 7})
        self.assertEqual(r.status_code, 400)

    def test_kpis_sem_token_da_401(self):
        self.assertEqual(self.cliente.get("/kpis").status_code, 401)

    def test_pedir_atendente_escala_na_hora(self):
        r = self.cliente.post("/mensagem", json={"texto": "quero falar com um atendente"})
        self.assertTrue(r.get_json()["transferido"])

    def test_auditoria_guarda_usuario_anonimizado(self):
        self.cliente.post("/mensagem", json={"texto": "oi", "usuario": "kadu"})
        self.assertNotEqual(dados["auditoria"][0]["usuario"], "kadu")

    def test_excluir_dados_do_usuario(self):
        self.cliente.post("/mensagem", json={"texto": "oi", "usuario": "kadu"})
        r = self.cliente.delete("/usuarios/kadu")
        self.assertEqual(r.status_code, 204)
        self.assertEqual(dados["auditoria"], [])

    def test_metodo_errado_da_405(self):
        self.assertEqual(self.cliente.delete("/status").status_code, 405)

    def test_calculo_dos_kpis(self):
        for _ in range(10):
            self.cliente.post("/visita")
        self.cliente.post("/mensagem", json={"texto": "oi", "usuario": "ana"})
        self.cliente.post("/mensagem", json={"texto": "tchau", "usuario": "ana"})
        self.cliente.post("/mensagem", json={"texto": "oi", "usuario": "bia"})
        self.cliente.post("/mensagem", json={"texto": "reembolso", "usuario": "bia"})
        self.cliente.post("/avaliacao", json={"nota": 5})
        self.cliente.post("/avaliacao", json={"nota": 4})

        k = self.cliente.get("/kpis", headers=TOKEN).get_json()
        self.assertEqual(k["engajamento"], 20.0)  # 2 usuários / 10 visitantes
        self.assertEqual(k["conclusao"], 75.0)    # 3 resolvidas / 4 interações
        self.assertEqual(k["transbordo"], 25.0)   # 1 transferida / 4 interações
        self.assertEqual(k["csat"], 4.5)          # (5 + 4) / 2


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest test_chatbot -v

### 3.1 Chamando as rotas da API (como se fosse o site ou o app)

In [ ]:
from chatbot_api import app

cliente = app.test_client()

r = cliente.post("/mensagem", json={"texto": "oi", "usuario": "kadu"})
print("POST /mensagem       ->", r.status_code, r.get_json())

r = cliente.post("/mensagem", json={"texto": "cadê meu pedido 1001?", "usuario": "kadu"})
print("POST /mensagem       ->", r.status_code, r.get_json())

r = cliente.post("/mensagem", json={})
print("POST /mensagem vazio ->", r.status_code, r.get_json())

r = cliente.get("/pedidos/9999")
print("GET /pedidos/9999    ->", r.status_code, r.get_json())

r = cliente.get("/kpis")
print("GET /kpis sem token  ->", r.status_code, r.get_json())

r = cliente.get("/kpis", headers={"Authorization": "Bearer segredo123"})
print("GET /kpis com token  ->", r.status_code, r.get_json())

## 4. Exercícios do professor no Colab

**Exercício 1 (turístico, OpenTripMap):** antes, baixe o modelo de português do spaCy e **reinicie a sessão**
(menu *Ambiente de execução → Reiniciar sessão*). Depois cole o código numa célula. A chave é pedida pelo `getpass`.

In [ ]:
!python -m spacy download pt_core_news_sm

**Exercício 2 (intenção, entidade e sentimento):** roda direto, sem instalar nada. É só colar numa célula.

> Dica: em qualquer chatbot com `while True` + `input()`, a célula fica rodando até você digitar `sair`.
> Se travar, use o botão ⏹ (interromper) da célula.